# 3.4 — Parquet vs CSV: plain text, gzip, and bzip2

**Chapter 3, section 3.7** (*Apache Parquet*), and Exercises 3, 4 and 11.

**The question this notebook answers:** section 3.7 claims three benefits for a columnar
format — column pruning, predicate pushdown, and per-column compression — and says their
combination produces "order-of-magnitude speed-ups". How many bytes does one realistic query
actually read under each layout, and how much of the reduction belongs to which mechanism?

The measurement is the point. A student should leave able to answer, of their own query on
their own data, *how many bytes does this read?* — and to attribute each reduction to a named
mechanism rather than to "Parquet is faster".

**Data.** GDELT 2.0 events, collected over the week of 2024-01-01 to 2024-01-07:
**731,884 rows × 61 columns**, about 290 MB as raw text. The width is what matters — a query
that wants five of sixty-one columns is a query with something to prune.

One property of the feed is worth knowing before the measurements, because it governs what the
date filter can do. `SQLDATE` is the date of the *event*, not of the collection, and GDELT
records events retrospectively, so a week of collected files carries 35 distinct event dates:
mostly the seven days of that week, with a long tail of older ones. That makes `SQLDATE` a
realistic partitioning column — low cardinality, very unevenly filled.

**Timings below are machine-dependent.** They came from one laptop running `local[*]`. The
byte counts are not machine-dependent and the ratios between them are the durable result.

Runs in about two minutes, most of it spent writing the copies of the table. A closing section
repeats the measurement against the full 65.8-million-row GDELT extract, and is skipped
automatically if the large Parquet file is not present. Both `.csv.gz` and `.csv.bz2`
are compared below: compression reduces stored bytes, while splittability determines whether
multiple tasks can decompress one file. Full scans of the large CSV copies are opt-in.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, shutil, tempfile, time, json, math
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType
import pandas as pd
import pyarrow.parquet as pq

DATA = os.environ.get("CS777_DATA", "../data")          # -> code/data/
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
WORK = os.path.join(SCRATCH, "ch03-parquet")            # the copies of the table live here
os.makedirs(WORK, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-3.4")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")   # keep printed output clean
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)
print("scratch:", WORK)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/08 22:22:43 WARN Utils: Your hostname, DT-MBPro.local, resolves to a loopback address: 127.0.0.1; using 192.168.10.156 instead (on interface en0)
26/09/08 22:22:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/08 22:22:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0
scratch: /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777/ch03-parquet


## The instruments, and what each one can honestly tell you

"How many bytes did that read?" has two answers here, and they measure different things. Both
are used below, and it is worth being clear about which is which — a measurement you cannot
interpret is worse than none.

**1. Spark's own scan metrics.** Every scan node publishes `number of files read` and `size of
files read`, and these are the numbers the Spark UI shows on the SQL tab. `number of files
read` is exactly what it says and is the honest measure of *partition pruning*. `size of files
read`, despite its name, is the size of the file *splits* assigned to tasks — the bytes the
planner handed out, not the bytes the reader fetched. It therefore **cannot see column
pruning**: ask for five of sixty-one columns and it reports the whole file anyway. Useful, and
easy to misread.

**2. Parquet footer arithmetic.** A Parquet file's footer records, for every row group and
every column chunk, its compressed size and its minimum and maximum value. The bytes a query
must fetch are therefore computable exactly: sum the compressed sizes of the wanted columns'
chunks, over the row groups whose min/max do not exclude the filter. The function below does
that with `pyarrow`, which is to say it reimplements column pruning and predicate pushdown from
the same footer Spark reads. Where CSV has no counterpart, its whole file must be read and
parsed to reach any field, so for CSV bytes-stored *is* bytes-scanned.

Wall-clock time comes from Spark in every case, because that is the only thing both formats can
be timed on.

*A note on units.* Every figure below labelled MB is a **mebibyte**, 1024², which is the unit
Spark's own metric formatter uses when it prints `44.3 MiB`. Sizes quoted in prose follow the
same convention, so that the numbers in the text and the numbers in the tables can be compared
without conversion.

In [2]:
SIZE_UNITS = {"B": 1, "KiB": 1024, "MiB": 1024**2, "GiB": 1024**3, "TiB": 1024**4}

def spark_scan_metrics():
    """`number of files read` / `size of files read` for the query that just ran --
    the same numbers the SQL tab of the Spark UI shows on the scan node."""
    store = spark._jsparkSession.sharedState().statusStore()
    execs = store.executionsList()
    last = execs.apply(execs.size() - 1)
    values, planned = store.executionMetrics(last.executionId()), last.metrics()
    seen = {}                                    # dedupe: AQE republishes the same accumulators
    for i in range(planned.size()):
        m = planned.apply(i)
        v = values.get(m.accumulatorId())
        if v.isDefined():
            seen[m.accumulatorId()] = (m.name(), v.get())
    files, split_bytes = 0, 0.0
    for name, text in seen.values():
        line = text.strip().split("\n")[-1].split("(")[0].strip()
        if name == "number of files read":
            files += int(line.replace(",", ""))
        elif name == "size of files read":
            num, unit = line.split()
            split_bytes += float(num.replace(",", "")) * SIZE_UNITS[unit]
    return {"files_read": files, "split_MB": round(split_bytes / 1024**2, 1)}

In [3]:
def parquet_files(root, only_dir=None):
    out = []
    for r, _, fs in os.walk(root):
        if only_dir and only_dir not in r:
            continue
        out += [os.path.join(r, f) for f in fs if f.endswith(".parquet")]
    return sorted(out)

def footer_scan(root, columns, predicate=None, only_dir=None):
    """Bytes a Parquet reader must fetch for `columns`, skipping any row group whose
    recorded min/max excludes `predicate` = (column, value).  This is column pruning
    and predicate pushdown, computed from the footer -- exactly what Spark does."""
    total, read, skipped = 0, 0, 0
    paths = parquet_files(root, only_dir)
    for path in paths:
        meta = pq.ParquetFile(path).metadata
        for g in range(meta.num_row_groups):
            rg = meta.row_group(g)
            names = [rg.column(j).path_in_schema for j in range(rg.num_columns)]
            if predicate:
                col, value = predicate
                stats = rg.column(names.index(col)).statistics
                if stats is not None and stats.has_min_max and not (stats.min <= value <= stats.max):
                    skipped += 1
                    continue
            read += 1
            for j, name in enumerate(names):
                if name in columns:
                    total += rg.column(j).total_compressed_size
    return {"MB_scanned": round(total / 1024**2, 2), "files_opened": len(paths),
            "rg_read": read, "rg_skipped": skipped}

def dir_bytes(path):
    if os.path.isfile(path):
        return os.path.getsize(path)
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs)

def mb(path):
    return round(dir_bytes(path) / 1024**2, 1)

## The data and its schema

The GDELT event table has 61 columns and no header row, so the schema must come from somewhere.
It ships with the data, in `gdelt-schema.txt`, one line per column giving position, name and
type. Declaring it is not optional here: with `inferSchema` Spark would read all 290 MB once to
guess the types and again to load — the doubled cost section 3.6 warns about — and would still
get some columns wrong.

In [4]:
SPARK_TYPES = {"string": StringType(), "integer": IntegerType(),
               "long": LongType(), "double": DoubleType()}

fields = []
with open(f"{DATA}/gdelt/gdelt-schema.txt") as fh:
    for line in fh:
        if line.startswith("#") or not line.strip():
            continue
        position, name, typ = line.rstrip("\n").split("\t")
        fields.append(StructField(name, SPARK_TYPES[typ], True))

GDELT_SCHEMA = StructType(fields)
print("columns declared:", len(GDELT_SCHEMA.fields))
print([f.name for f in GDELT_SCHEMA.fields[:6]], "...", [f.name for f in GDELT_SCHEMA.fields[-2:]])

columns declared: 61
['GLOBALEVENTID', 'SQLDATE', 'MonthYear', 'Year', 'FractionDate', 'Actor1Code'] ... ['DATEADDED', 'SOURCEURL']


In [5]:
src = (spark.read
       .option("sep", "\t")
       .option("header", "false")
       .schema(GDELT_SCHEMA)                     # declared, not inferred
       .csv(f"{DATA}/gdelt/gdelt-events-small.csv.gz"))

print("rows:", src.count())
src.select("GLOBALEVENTID", "SQLDATE", "ActionGeo_CountryCode", "AvgTone").show(5)

days = src.groupBy("SQLDATE").count().orderBy(F.desc("count"))
print("distinct event dates:", days.count())
days.show(8)

rows: 731884
+-------------+--------+---------------------+-----------------+
|GLOBALEVENTID| SQLDATE|ActionGeo_CountryCode|          AvgTone|
+-------------+--------+---------------------+-----------------+
|   1149157552|20230101|                   US|-1.02974828375286|
|   1149157553|20230101|                   CH|             -0.8|
|   1149157554|20230101|                   FR|              0.0|
|   1149157555|20230101|                   JA|              0.0|
|   1149157556|20230101|                   FR|              0.0|
+-------------+--------+---------------------+-----------------+
only showing top 5 rows


distinct event dates: 35


+--------+------+
| SQLDATE| count|
+--------+------+
|20240104|133599|
|20240105|128265|
|20240103|121139|
|20240102|104777|
|20240106| 85988|
|20240107| 74272|
|20240101| 66403|
|20230104|  1680|
+--------+------+
only showing top 8 rows


## The same table, written four ways

Same 731,884 rows, same 61 columns, four physical layouts:

1. **`csv/`** — plain uncompressed CSV. The baseline: bytes stored equals bytes scanned.
2. **`parquet/`** — one Snappy-compressed Parquet file, rows in arbitrary order.
3. **`parquet_sorted/`** — the same single file, but the rows **sorted by `SQLDATE`** before
   writing. Copies 2 and 3 are written to one file each so that the *only* difference between
   them is the row order; a different file count would confound the comparison, because file
   boundaries can exclude a date just as row-group boundaries can.
4. **`parquet_by_day/`** — Parquet **partitioned by `SQLDATE`**, one file per date, so the date
   appears in the directory name rather than in the data.

Copies 2 and 3 exist to separate two things the chapter treats together. Both are Parquet, both
carry footer statistics, both are read by the same code. Only the sorted one has its dates
concentrated into distinct row groups. The gap between them is the whole of predicate pushdown.

One deliberate departure from the conventional settings: the Parquet copies are written with an
**8 MB row group** rather than the customary 128 MB. The row group is the unit at which
pushdown skips data, and 128 MB over a 90 MB table would mean one row group per file and
nothing to skip. Section 3.7.3 makes exactly this point — the row-group size *governs the
granularity of the skipping* — and it is measured directly at the end of this notebook.

In [6]:
ROW_GROUP = 8 * 1024 * 1024                    # 8 MB; see the note above

CSV_DIR   = os.path.join(WORK, "csv")
PQ_DIR    = os.path.join(WORK, "parquet")
PQ_SORTED = os.path.join(WORK, "parquet_sorted")
PQ_DAY    = os.path.join(WORK, "parquet_by_day")

if not os.path.exists(PQ_DAY):                 # a minute or so; skipped on later runs
    cached = src.repartition(8).cache()
    cached.count()

    t0 = time.time()
    cached.write.mode("overwrite").option("header", "true").csv(CSV_DIR)
    print(f"{'CSV':22s} {time.time()-t0:5.1f}s")

    t0 = time.time()
    (cached.coalesce(1)                         # one file, rows in arbitrary order
           .write.mode("overwrite")
           .option("parquet.block.size", ROW_GROUP).parquet(PQ_DIR))
    print(f"{'Parquet':22s} {time.time()-t0:5.1f}s")

    t0 = time.time()
    (cached.sort("SQLDATE").coalesce(1)         # one file, globally sorted by date
           .write.mode("overwrite")
           .option("parquet.block.size", ROW_GROUP).parquet(PQ_SORTED))
    print(f"{'Parquet sorted':22s} {time.time()-t0:5.1f}s")

    t0 = time.time()
    (cached.repartition("SQLDATE")             # by the column: one file per date
           .write.mode("overwrite")
           .option("parquet.block.size", ROW_GROUP)
           .partitionBy("SQLDATE").parquet(PQ_DAY))
    print(f"{'Parquet by day':22s} {time.time()-t0:5.1f}s")

    cached.unpersist()

sizes = pd.DataFrame([
    {"layout": "gz CSV (as shipped)", "on disk (MB)": mb(f"{DATA}/gdelt/gdelt-events-small.csv.gz"), "files": 1},
    {"layout": "bz2 CSV (as shipped)", "on disk (MB)": mb(f"{DATA}/gdelt/gdelt-events-small.csv.bz2"), "files": 1},
    {"layout": "CSV, uncompressed",   "on disk (MB)": mb(CSV_DIR),
     "files": sum(1 for f in os.listdir(CSV_DIR) if f.endswith(".csv"))},
    {"layout": "Parquet",             "on disk (MB)": mb(PQ_DIR),    "files": len(parquet_files(PQ_DIR))},
    {"layout": "Parquet, sorted",     "on disk (MB)": mb(PQ_SORTED), "files": len(parquet_files(PQ_SORTED))},
    {"layout": "Parquet, by day",     "on disk (MB)": mb(PQ_DAY),    "files": len(parquet_files(PQ_DAY))},
])
csv_size = sizes.loc[sizes["layout"] == "CSV, uncompressed", "on disk (MB)"].iloc[0]
sizes["vs CSV"] = (sizes["on disk (MB)"] / csv_size).round(3)
print(sizes.to_string(index=False))

              layout  on disk (MB)  files  vs CSV
 gz CSV (as shipped)          44.3      1   0.152
bz2 CSV (as shipped)          27.3      1   0.094
   CSV, uncompressed         290.5      8   1.000
             Parquet         100.7      1   0.347
     Parquet, sorted          88.1      1   0.303
     Parquet, by day          88.1     35   0.303


The compression ratio alone is the first surprise, and general-purpose compression is not what
earns it. Parquet stores each column's values contiguously and encodes each column according to
what it holds — run-length encoding for long runs of a repeated value, dictionary encoding for
a column drawn from a small set of distinct values — *before* Snappy sees a byte. A column of
country codes with a few hundred distinct values costs a dictionary plus one small integer per
row. The same column as text costs two characters and a delimiter, every row, forever.

Note also that the sorted copy is smaller than the unsorted one, from nothing but the ordering:
sorting brings like values together, and like values adjacent are what run-length and
dictionary encodings are built to exploit.

## One realistic query

Five columns out of sixty-one; one date out of thirty-five; grouped down to a couple of hundred
rows. This is deliberately the shape of Exercise 4, and of the query Exercise 11 asks for
against the taxi data.

> *For one day, how many events were recorded in each country, and what was their average
> tone?*

In [7]:
DAY = 20240104                                 # the busiest date in the sample
QUERY_COLUMNS = {"SQLDATE", "ActionGeo_CountryCode", "AvgTone", "NumMentions", "EventRootCode"}

def query(df, with_filter=True):
    q = df.where(F.col("SQLDATE") == DAY) if with_filter else df
    return (q.select(*sorted(QUERY_COLUMNS))
             .groupBy("ActionGeo_CountryCode")
             .agg(F.count("*").alias("events"),
                  F.round(F.avg("AvgTone"), 2).alias("avg_tone"),
                  F.sum("NumMentions").alias("mentions"))
             .orderBy(F.desc("events")))

# What the answer looks like, so that the rest of the notebook is measuring something real.
query(spark.read.parquet(PQ_DIR)).show(8)

+---------------------+------+--------+--------+
|ActionGeo_CountryCode|events|avg_tone|mentions|
+---------------------+------+--------+--------+
|                   US| 46131|   -2.61|  205577|
|                   IS|  9289|   -5.51|   36437|
|                   UK|  7333|   -2.23|   37253|
|                   IN|  5896|   -1.34|   30451|
|                 NULL|  3693|   -2.13|   34240|
|                   NI|  3336|   -2.03|   15228|
|                   IR|  3177|   -7.27|   13921|
|                   CA|  3057|   -1.65|   16218|
+---------------------+------+--------+--------+
only showing top 8 rows


## The measurement

Six layout runs compare filtering, column pruning, and row-group or directory pruning.
The compressed-text comparison that follows separates storage size from read parallelism.

In [8]:
csv_df    = (spark.read.option("header", "true").option("sep", ",")
             .schema(GDELT_SCHEMA).csv(CSV_DIR))
pq_df     = spark.read.parquet(PQ_DIR)
pqsort_df = spark.read.parquet(PQ_SORTED)
pqday_df  = spark.read.parquet(PQ_DAY)

RESULTS = []

def run(label, df, with_filter, scanned, mechanism):
    t0 = time.time()
    rows = query(df, with_filter).collect()
    seconds = time.time() - t0
    m = spark_scan_metrics()
    RESULTS.append({"query": label,
                    "seconds": round(seconds, 2),
                    "MB scanned": scanned["MB_scanned"],
                    "files": m["files_read"],
                    "row groups": f"{scanned.get('rg_read', '-')}/"
                                  f"{scanned.get('rg_read', 0) + scanned.get('rg_skipped', 0)}",
                    "rows out": len(rows),
                    "mechanism": mechanism})

csv_all = {"MB_scanned": mb(CSV_DIR)}          # a text reader must read every byte to find a field

run("1. CSV, no filter",         csv_df,    False, csv_all, "baseline: reads everything")
run("2. CSV, day filter",        csv_df,    True,  csv_all, "filter applies after the read")
run("3. Parquet, no filter",     pq_df,     False,
    footer_scan(PQ_DIR, QUERY_COLUMNS), "column pruning")
run("4. Parquet, day filter",    pq_df,     True,
    footer_scan(PQ_DIR, QUERY_COLUMNS, predicate=("SQLDATE", DAY)),
    "pruning; pushdown cannot fire")
run("5. Parquet sorted, filter", pqsort_df, True,
    footer_scan(PQ_SORTED, QUERY_COLUMNS, predicate=("SQLDATE", DAY)),
    "pruning + row-group skipping")
run("6. Parquet by day, filter", pqday_df,  True,
    footer_scan(PQ_DAY, QUERY_COLUMNS, only_dir=f"SQLDATE={DAY}"),
    "pruning + partition pruning")

table = pd.DataFrame(RESULTS)
table["vs row 1"] = (table.loc[0, "MB scanned"] / table["MB scanned"]).round(1).astype(str) + "x less"
print(table.to_string(index=False))

                    query  seconds  MB scanned  files row groups  rows out                     mechanism    vs row 1
        1. CSV, no filter     0.74      290.50      8        -/0       242    baseline: reads everything   1.0x less
       2. CSV, day filter     0.43      290.50      8        -/0       223 filter applies after the read   1.0x less
    3. Parquet, no filter     0.40        6.51      1      21/21       242                column pruning  44.6x less
   4. Parquet, day filter     0.29        6.51      1      21/21       223 pruning; pushdown cannot fire  44.6x less
5. Parquet sorted, filter     0.15        1.24      1       5/20       223  pruning + row-group skipping 234.3x less
6. Parquet by day, filter     0.18        0.89      1        4/4       223   pruning + partition pruning 326.4x less


### Reading the table

**Rows 1 → 2: the filter buys nothing.** Both CSV runs scan the same bytes. There is nowhere in
a text file to record that rows 400,000 to 500,000 all belong to 4 January, so the only way to
find out is to read and parse them. In a row-oriented format a filter can only be applied
*after* the read: it saves downstream work and not one byte of I/O.

**Rows 1 → 3: column pruning.** No filter at all, and the scan collapses by well over an order
of magnitude. Five of sixty-one columns are wanted, and the reader opens exactly those five
column chunks and never touches the other fifty-six. Two effects compound here and it is worth
keeping them apart: the whole table is about three times smaller as Parquet (encoding and
compression, visible in the sizes table above), and of that smaller table only a small fraction
of the columns is opened (pruning). This is the distinction between bytes stored and bytes
scanned drawn at the very start of the course, now realized in a file layout.

**Rows 3 → 4: pushdown that does not fire.** This is the row worth dwelling on, because it
disappoints. The plan below shows `PushedFilters: [EqualTo(SQLDATE,20240104)]` — Catalyst did
push the filter down to the scan — and not a single row group is skipped. The footer records
each chunk's minimum and maximum `SQLDATE`, and a row group can be skipped only when those
bounds *exclude* the wanted value. These rows were shuffled before writing, so every row group
holds a scattering of every date, every min/max straddles the target, and nothing is
excludable.

**Rows 4 → 5: pushdown that does fire.** Same format, same footer statistics, same filter, same
reader. The only difference is that the rows were sorted by `SQLDATE` before writing, so each
date is concentrated into a few row groups instead of smeared across all of them. Now most
groups' bounds exclude the target and are skipped unread — see the `row groups` column, which
reports groups-read over groups-total. *The format makes the mechanism possible; the write
decides whether it fires.* That sentence is worth more than any single number in the table.

**Rows 5 → 6: partition pruning.** With the date in the directory name, Spark performs partition
discovery from the directory listing alone and never opens the other thirty-four dates' files at
all — note the `files` column, which comes from Spark's own metric rather than from the footer
arithmetic. This is coarser and cheaper than footer statistics: it needs no statistics, and no
file is opened even to be rejected. It is also the mechanism most easily abused, which is the
small-files section below.

## Gzip vs bzip2 vs Parquet: compression is not column pruning

The two shipped compressed CSV files contain identical tab-separated text. Gzip is not
splittable; bzip2 is, so Spark can assign different blocks of one `.bz2` file to different
tasks. Both must still decompress and examine the entire text to answer the day query.
Parquet can avoid unneeded column chunks and, where statistics permit, row groups.

For this small-file demonstration, set `spark.sql.files.maxPartitionBytes` to **8 MiB** for
all three readers, then restore it. Otherwise a 27 MiB bzip2 file may get only one task at
the usual 128 MiB setting, hiding its splittability. This is a demonstration, not a tuning
recommendation: extra tasks and bzip2 decoding can cost more than parallelism saves.

Each query runs once to warm up and twice for timing; report the best timed run. The table
distinguishes **compressed bytes stored/read** for CSV from **raw text decoded**. Parquet's
scan figure is a footer-based estimate of compressed column-chunk bytes, excluding metadata
and reader overhead, not a measured I/O counter. Only four columns survive optimization:
`EventRootCode` is selected by `query()` but unused by its aggregation.


In [9]:
with open(f"{DATA}/gdelt/manifest.json") as fh:
    manifest = json.load(fh)

def read_gdelt_csv(path):
    return (spark.read.option("sep", "\t").option("header", "false")
            .schema(GDELT_SCHEMA).csv(path))

def assert_same_results(actual, expected):
    actual = {r.ActionGeo_CountryCode: r.asDict() for r in actual}
    expected = {r.ActionGeo_CountryCode: r.asDict() for r in expected}
    assert actual.keys() == expected.keys(), "Country groups differ between formats"
    for country, row in expected.items():
        for name, value in row.items():
            other = actual[country][name]
            if isinstance(value, float):
                assert other is not None and math.isclose(other, value, rel_tol=1e-9, abs_tol=1e-9), (country, name)
            else:
                assert other == value, (country, name)

SMALL_GZ = f"{DATA}/gdelt/gdelt-events-small.csv.gz"
SMALL_BZ2 = f"{DATA}/gdelt/gdelt-events-small.csv.bz2"
SCAN_COLUMNS = QUERY_COLUMNS - {"EventRootCode"}
original_split_size = spark.conf.get("spark.sql.files.maxPartitionBytes")
compressed_rows = []
try:
    spark.conf.set("spark.sql.files.maxPartitionBytes", 8 * 1024**2)
    reference = query(spark.read.parquet(PQ_DIR)).collect()
    for label, path in [("CSV gzip", SMALL_GZ), ("CSV bzip2", SMALL_BZ2), ("Parquet", PQ_DIR)]:
        df = spark.read.parquet(path) if label == "Parquet" else read_gdelt_csv(path)
        partitions = df.rdd.getNumPartitions()
        assert_same_results(query(df).collect(), reference)
        timings = []
        for _ in range(2):
            t0 = time.perf_counter()
            result = query(df).collect()
            timings.append(time.perf_counter() - t0)
            assert_same_results(result, reference)
        scanned = (footer_scan(path, SCAN_COLUMNS, predicate=("SQLDATE", DAY))["MB_scanned"]
                   if label == "Parquet" else mb(path))
        compressed_rows.append({"format": label, "stored MB": mb(path),
                                "scan MB (estimate)": scanned,
                                "raw text MB decoded": "-" if label == "Parquet" else round(manifest["small"]["raw_bytes"] / 1024**2, 1),
                                "read partitions": partitions, "best seconds": round(min(timings), 2),
                                "same answer": True})
finally:
    spark.conf.set("spark.sql.files.maxPartitionBytes", original_split_size)

print(pd.DataFrame(compressed_rows).to_string(index=False))
print("Bzip2 can parallelize decompression; it does not enable column or row-group pruning.")

   format  stored MB  scan MB (estimate) raw text MB decoded  read partitions  best seconds  same answer
 CSV gzip       44.3               44.30               284.3                1          1.22         True
CSV bzip2       27.3               27.30               284.3                7          1.18         True
  Parquet      100.7                6.07                   -               18          0.28         True
Bzip2 can parallelize decompression; it does not enable column or row-group pruning.


## The evidence in the plan

None of this need be taken on trust either. `explain()` prints the physical plan, and on a
Parquet source it reports two entries directly: **`PushedFilters`**, listing the conditions that
reached the scan, and **`ReadSchema`**, naming the columns actually read. Those two are the
answer to Exercise 3.

In [10]:
print("=== Parquet source ===")
query(pq_df).explain()

=== Parquet source ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [events#1469L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(events#1469L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1793]
      +- HashAggregate(keys=[ActionGeo_CountryCode#438], functions=[count(1), avg(AvgTone#419), sum(NumMentions#416)])
         +- Exchange hashpartitioning(ActionGeo_CountryCode#438, 200), ENSURE_REQUIREMENTS, [plan_id=1790]
            +- HashAggregate(keys=[ActionGeo_CountryCode#438], functions=[partial_count(1), partial_avg(AvgTone#419), partial_sum(NumMentions#416)])
               +- Project [ActionGeo_CountryCode#438, AvgTone#419, NumMentions#416]
                  +- Filter (isnotnull(SQLDATE#386) AND (SQLDATE#386 = 20240104))
                     +- FileScan parquet [SQLDATE#386,NumMentions#416,AvgTone#419,ActionGeo_CountryCode#438] Batched: true, DataFilters: [isnotnull(SQLDATE#386), (SQLDATE#386 = 20240104)], Format: Parquet, Location: InMemoryFil

In [11]:
print("=== CSV source: same query, same optimizer ===")
query(csv_df).explain()

=== CSV source: same query, same optimizer ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [events#1488L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(events#1488L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1817]
      +- HashAggregate(keys=[ActionGeo_CountryCode#377], functions=[count(1), avg(AvgTone#358), sum(NumMentions#355)])
         +- Exchange hashpartitioning(ActionGeo_CountryCode#377, 200), ENSURE_REQUIREMENTS, [plan_id=1814]
            +- HashAggregate(keys=[ActionGeo_CountryCode#377], functions=[partial_count(1), partial_avg(AvgTone#358), partial_sum(NumMentions#355)])
               +- Project [ActionGeo_CountryCode#377, AvgTone#358, NumMentions#355]
                  +- Filter (isnotnull(SQLDATE#325) AND (SQLDATE#325 = 20240104))
                     +- FileScan csv [SQLDATE#325,NumMentions#355,AvgTone#358,ActionGeo_CountryCode#377] Batched: false, DataFilters: [isnotnull(SQLDATE#325), (SQLDATE#325 = 20240104)], Format: CSV, Loca

Both plans carry a `PushedFilters` entry — Catalyst pushes the filter as far down as the source
will accept it in either case. What differs is what the source can *do* with it. For Parquet,
`PushedFilters: [IsNotNull(SQLDATE), EqualTo(SQLDATE,20240104)]` reaches a reader that can
compare it against footer statistics and skip whole row groups. The CSV reader can only apply
the same predicate to rows it has already read and parsed, so the pushdown appears in the plan
and saves no I/O — which is exactly what rows 1 and 2 of the table measured. A `PushedFilters`
entry is evidence that the filter *reached* the scan, not that it *saved* anything.

`ReadSchema` is the unambiguous one: it names four or five columns rather than sixty-one, in
both plans. For Parquet that is a promise the layout can keep; for CSV the reader must still
tokenize every line to reach them.

On the partitioned layout the date disappears from `PushedFilters` altogether and appears as a
**`PartitionFilters`** entry instead — the filter was answered from the directory names before
any file was opened.

In [12]:
print("=== Partitioned Parquet: note PartitionFilters, and the empty PushedFilters ===")
query(pqday_df).explain()

=== Partitioned Parquet: note PartitionFilters, and the empty PushedFilters ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [events#1507L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(events#1507L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1839]
      +- HashAggregate(keys=[ActionGeo_CountryCode#559], functions=[count(1), avg(AvgTone#540), sum(NumMentions#537)])
         +- Exchange hashpartitioning(ActionGeo_CountryCode#559, 200), ENSURE_REQUIREMENTS, [plan_id=1836]
            +- HashAggregate(keys=[ActionGeo_CountryCode#559], functions=[partial_count(1), partial_avg(AvgTone#540), partial_sum(NumMentions#537)])
               +- Project [ActionGeo_CountryCode#559, AvgTone#540, NumMentions#537]
                  +- FileScan parquet [NumMentions#537,AvgTone#540,ActionGeo_CountryCode#559,SQLDATE#567] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777/

## Writing Parquet well, 1: the row-group size

Section 3.7.3 names three write-time choices, and the row group is the second. It is the unit at
which pushdown skips data, so it sets the granularity of the skipping: a large row group
compresses better and reads sequentially faster but coarsens the skipping, since a filter must
read a whole group to reach the few rows in it that match.

Here is that trade-off measured. The same sorted rows, written at three row-group sizes, with
the same filtered query costed from the footer.

In [13]:
rg_rows = []
for label, size in [("128 MB (conventional)", 128 * 1024**2),
                    ("8 MB",                    8 * 1024**2),
                    ("1 MB",                    1 * 1024**2)]:
    path = os.path.join(WORK, f"rg_{size}")
    if not os.path.exists(path):
        # built exactly like parquet_sorted/ above, so the row-group size is the only
        # variable and these numbers are comparable with the results table
        (src.repartition(8).sort("SQLDATE").coalesce(1)
            .write.mode("overwrite").option("parquet.block.size", size).parquet(path))
    s = footer_scan(path, QUERY_COLUMNS, predicate=("SQLDATE", DAY))
    rg_rows.append({"row group": label, "size on disk (MB)": mb(path),
                    "row groups": s["rg_read"] + s["rg_skipped"],
                    "read": s["rg_read"], "skipped": s["rg_skipped"],
                    "MB scanned": s["MB_scanned"]})

print(pd.DataFrame(rg_rows).to_string(index=False))

            row group  size on disk (MB)  row groups  read  skipped  MB scanned
128 MB (conventional)               85.7           1     1        0        3.58
                 8 MB               88.1          20     5       15        1.24
                 1 MB              119.1         314    58      256        1.32


At 128 MB the whole table is a single row group. It contains every date, its recorded minimum
and maximum `SQLDATE` therefore span everything, and nothing whatever can be skipped — the
query reads all five columns in full. Dropping to 8 MB gives twenty row groups, fifteen of
which the filter excludes, and the bytes scanned fall by roughly two thirds.

Then look at 1 MB, and notice that it is *worse*. The skipping is finer still — 256 of 314
groups excluded — and yet slightly more data is scanned, from a file 35% larger on disk. Each
row group carries its own footer entry and its own dictionaries, and cutting the data into
smaller pieces shortens the runs of repeated values that the encoders exist to exploit. Past
some point the per-group overhead outgrows the sharper skipping.

That is the whole shape of the trade-off, and it is why the conventional figure is a
convention rather than a rule. It is a balance struck for tables far larger than this one,
where a 128 MB row group is already a small fraction of the data. The lesson is not "use 8 MB
row groups"; it is that the right figure is a *ratio* — to the size of the table, and to how
selective the filters you expect are going to be.

## Writing Parquet well, 2: the compression codec

`snappy` by default, chosen for balance; `zstd` or `gzip` for smaller files at more processor
time. The consideration is that decompression happens on *every* read, so a codec that saves
storage at the expense of decode speed is paid for by every query thereafter.

In [14]:
def decode_bound_read(path):
    """A read the codec has to work for: every row, twelve columns, no filter."""
    cols = [f.name for f in GDELT_SCHEMA.fields[:12]]
    return spark.read.parquet(path).select(*cols).agg(*[F.count(c) for c in cols]).collect()

codec_rows = []
for codec in ["snappy", "zstd", "gzip"]:
    path = os.path.join(WORK, f"codec_{codec}")
    if not os.path.exists(path):
        src.repartition(4).write.mode("overwrite").option("compression", codec).parquet(path)
    decode_bound_read(path)                     # warm the page cache and the JIT
    t0 = time.time()
    decode_bound_read(path)                     # the timing that is reported
    codec_rows.append({"codec": codec, "size (MB)": mb(path), "read (s)": round(time.time() - t0, 2)})

codecs = pd.DataFrame(codec_rows)
codecs["size vs snappy"] = (codecs["size (MB)"] / codecs.loc[0, "size (MB)"]).round(3)
print(codecs.to_string(index=False))
print("\n(seconds on one laptop; read the sizes, treat the seconds with suspicion)")

 codec  size (MB)  read (s)  size vs snappy
snappy       93.3      0.16           1.000
  zstd       62.4      0.08           0.669
  gzip       66.9      0.09           0.717

(seconds on one laptop; read the sizes, treat the seconds with suspicion)


Read the size column first, and be careful with the seconds. The ordering of the *sizes* is a
property of the codecs and of this data, and it is what should drive the choice: `zstd` is
substantially smaller than Snappy here, with `gzip` between them. The read seconds, on a
730,000-row table on one laptop, are close enough together to be as much measurement noise as
signal — decode simply is not the bottleneck at this scale.

That is itself the lesson. At small scale the codec barely matters, and by the time it does
matter you should be measuring on your own data rather than trusting a table in a notebook. The
general shape, which does hold at scale, is that `zstd` gives most of `gzip`'s compression at
closer to Snappy's decode speed, and that a codec chosen for storage is paid for by every query
thereafter.

## Writing Parquet well, 3: the small-files problem

The third choice, and the most common error, concerns the *number* of files rather than their
contents. Every file carries a fixed cost — open it, read its footer, plan around it, schedule
a task for it — incurred whether it holds a gigabyte or a kilobyte. Here is the same data
written twice: once across 200 fragments, once across two.

In [15]:
MANY = os.path.join(WORK, "many_files")
FEW  = os.path.join(WORK, "few_files")

if not os.path.exists(FEW):
    src.repartition(200).write.mode("overwrite").parquet(MANY)
    # `coalesce` reduces a partition count without a shuffle, and is the usual remedy; the
    # source here is a single gzip stream and so already has one partition, hence `repartition`.
    src.repartition(2).write.mode("overwrite").parquet(FEW)

small_rows = []
for label, path in [("200 files", MANY), ("2 files", FEW)]:
    n = len(parquet_files(path))
    t0 = time.time()
    query(spark.read.parquet(path)).collect()
    small_rows.append({"layout": label, "files": n, "total size (MB)": mb(path),
                       "avg file (KB)": round(dir_bytes(path) / n / 1024),
                       "MB scanned": footer_scan(path, QUERY_COLUMNS,
                                                 predicate=("SQLDATE", DAY))["MB_scanned"],
                       "query (s)": round(time.time() - t0, 2)})

print(pd.DataFrame(small_rows).to_string(index=False))

   layout  files  total size (MB)  avg file (KB)  MB scanned  query (s)
200 files    200            122.6            628        7.46       0.46
  2 files      2             91.2          46670        4.46       0.15


Same rows, same columns, same query — and the 200-file version is both *larger on disk* and
slower to read. Larger, because each file carries its own footer and its own dictionaries, and
splitting the data two hundred ways destroys the long runs of one column's values that the
encoders exist to exploit. Slower, because the reader must open, plan around, and schedule a
task for every fragment.

The same trap is what makes `partitionBy` on a high-cardinality column so damaging: partition
by user ID and you get one directory, and at least one file, per user. Reserve partitioning for
columns whose distinct values number in the tens or hundreds — a date, a region — and control
the file count deliberately, with `coalesce` before writing, or `repartition` where a shuffle is
acceptable.

---

## Does any of this hold at scale?

A fair objection to everything above: 731,884 rows is 290 MB, which is not big data by any
definition, and ratios measured on a table that fits in memory may be an artefact of the toy.

So here is the same query, on the same 61 columns, against the **full** GDELT extract that ships
with this course: **65,800,035 rows** — ninety times as many — held as
`gdelt-events-large.parquet` (4.7 GiB), `gdelt-events-large.csv.gz` (3.9 GiB compressed),
and `gdelt-events-large.csv.bz2` (2.4 GiB compressed). Both CSV copies expand to the same
**25 GiB** of raw text.

Two things make this more than a bigger version of the same demonstration.

First, CSV timings are **opt-in**: both codecs must decode 25 GiB, despite their different
read parallelism. Set `CS777_RUN_LARGE_CSV=1` before running the notebook to time both.

Second, this file settles the sorted-versus-shuffled question on real data rather than on a
copy re-written to make a point. GDELT rows arrive in chronological order of `DATEADDED`, the
moment the event was *collected*, and that is the order they are stored in. But `SQLDATE`, the
date the event *happened*, is retrospective: any given batch contains events dated years
earlier. So one file carries a column whose values are tightly clustered per row group and
another whose values span a decade in every row group — and the same query filtered on each
gives the two halves of the pushdown lesson, from one physical file.

In [16]:
LARGE_PARQUET = f"{DATA}/gdelt/gdelt-events-large.parquet"
LARGE_CSV_GZ  = f"{DATA}/gdelt/gdelt-events-large.csv.gz"
LARGE_CSV_BZ2 = f"{DATA}/gdelt/gdelt-events-large.csv.bz2"
RUN_LARGE_CSV = os.environ.get("CS777_RUN_LARGE_CSV", "0") == "1"
HAVE_LARGE = os.path.exists(LARGE_PARQUET)

if not HAVE_LARGE:
    print("The large GDELT extract is not present in code/data/gdelt/, so this section is")
    print("skipped.  Everything above ran on the small extract and is unaffected.")
else:
    meta = pq.ParquetFile(LARGE_PARQUET).metadata          # footer only: no data is read
    print(f"rows        {meta.num_rows:,}")
    print(f"columns     {meta.num_columns}")
    print(f"row groups  {meta.num_row_groups}  (1 M rows each)")
    print(f"parquet     {mb(LARGE_PARQUET):>10,.1f} MB on disk")
    print(f"csv.gz      {mb(LARGE_CSV_GZ):>10,.1f} MB on disk")
    print(f"csv.bz2     {mb(LARGE_CSV_BZ2):>10,.1f} MB on disk")
    print("Time full CSV scans:", RUN_LARGE_CSV)

rows        65,800,035
columns     61
row groups  125  (1 M rows each)
parquet        4,831.6 MB on disk
csv.gz         4,014.2 MB on disk
csv.bz2        2,473.1 MB on disk
Time full CSV scans: False


### Splittability: gzip, bzip2, and Parquet

A `gzip` stream cannot be decompressed from the middle, so a `.gz` file is **not splittable**:
Spark must hand the whole thing to one task, on one core, however many cores it has. The
`.bz2` copy *is* splittable, as is Parquet. That is visible without executing a full scan,
straight from the planned read-partition counts. The original split-size setting has been
restored after the small-file experiment; the large bzip2 file is big enough to split under
the usual 128 MiB setting.

In [17]:
if HAVE_LARGE:
    gz = read_gdelt_csv(LARGE_CSV_GZ)
    bz = read_gdelt_csv(LARGE_CSV_BZ2)
    pqt = spark.read.parquet(LARGE_PARQUET)
    print(f"cores available to this session : {spark.sparkContext.defaultParallelism}")
    print(f"partitions for the 4.2 GB .gz   : {gz.rdd.getNumPartitions()}")
    print(f"partitions for the .bz2        : {bz.rdd.getNumPartitions()}")
    print(f"partitions for the 5.1 GB parquet: {pqt.rdd.getNumPartitions()}")

cores available to this session : 18
partitions for the 4.2 GB .gz   : 1
partitions for the .bz2        : 20
partitions for the 5.1 GB parquet: 38


Gzip has one read partition: one task decompresses 25 GiB. Bzip2 can distribute that work
across tasks, but each record still has to be decoded and examined. Splittability removes
the one-task restriction; it does not guarantee a faster query or reduce the raw text processed.

The first table below retains **25,609 MB of decoded CSV text** as a logical baseline from
`manifest.json`. This is not compressed disk I/O: gzip reads about 4,014 MB and bzip2 about
2,473 MB. Its Parquet figures sum all five selected columns, an upper bound because Catalyst
can prune selected columns unused by the final aggregation. The second table uses the two
columns actually needed by the no-filter aggregation and compares compressed scan estimates,
read partitions, and optional timings directly. Neither CSV codec offers Parquet's column
or row-group pruning.

In [18]:
if HAVE_LARGE:
    LARGE_COLS = {"SQLDATE", "DATEADDED", "ActionGeo_CountryCode", "AvgTone", "NumMentions"}

    def large_footer(predicate=None):
        """Same footer arithmetic as before, over the 125 row groups of the large file."""
        total, read, skipped = 0, 0, 0
        for g in range(meta.num_row_groups):
            rg = meta.row_group(g)
            names = [rg.column(j).path_in_schema for j in range(rg.num_columns)]
            if predicate:
                col, lo, hi = predicate
                st = rg.column(names.index(col)).statistics
                if st is not None and st.has_min_max and (st.max < lo or st.min > hi):
                    skipped += 1
                    continue
            read += 1
            for j, name in enumerate(names):
                if name in LARGE_COLS:
                    total += rg.column(j).total_compressed_size
        return total / 1024**2, read, skipped

    def large_query(where=None, source=None):
        d = pqt if source is None else source
        d = d.where(where) if where is not None else d
        return (d.select(*sorted(LARGE_COLS))
                 .groupBy("ActionGeo_CountryCode")
                 .agg(F.count("*").alias("events"), F.avg("AvgTone").alias("avg_tone")))

    DAY_LO, DAY_HI = 20240601000000, 20240601235959      # DATEADDED is YYYYMMDDHHMMSS
    CSV_MB = manifest["large"]["raw_bytes"] / 1024**2

    large_runs = [
        ("CSV gzip/bzip2, decoded text", None, None, "both decode all text; not disk I/O"),
        ("Parquet, no filter",                  None, None, "column pruning: 5 of 61"),
        ("Parquet, filter on DATEADDED",
         (F.col("DATEADDED") >= DAY_LO) & (F.col("DATEADDED") <= DAY_HI),
         ("DATEADDED", DAY_LO, DAY_HI), "pushdown on the SORTED column"),
        ("Parquet, filter on SQLDATE",
         F.col("SQLDATE") == 20240601, ("SQLDATE", 20240601, 20240601),
         "pushdown on an UNSORTED column"),
    ]

    large_rows = []
    for label, where, predicate, mechanism in large_runs:
        if label.startswith("CSV"):
            large_rows.append({"query": label, "seconds": "not run", "payload MB": round(CSV_MB, 1),
                               "row groups": "-", "vs raw CSV": "baseline", "mechanism": mechanism})
            continue
        scanned, read, skipped = large_footer(predicate)
        t0 = time.time()
        large_query(where).collect()
        large_rows.append({"query": label, "seconds": f"{time.time() - t0:.2f}",
                           "payload MB": round(scanned, 1),
                           "row groups": f"{read}/{read + skipped}",
                           "vs raw CSV": f"{CSV_MB / scanned:,.0f}x less",
                           "mechanism": mechanism})

    print(pd.DataFrame(large_rows).to_string(index=False))

    # Match the no-filter query's actual scan: unused selected columns are pruned by Catalyst.
    no_filter_columns = {"ActionGeo_CountryCode", "AvgTone"}
    parquet_scan_mb = sum(meta.row_group(g).column(j).total_compressed_size
                          for g in range(meta.num_row_groups)
                          for j in range(meta.num_columns)
                          if meta.row_group(g).column(j).path_in_schema in no_filter_columns) / 1024**2
    format_rows = []
    reference = None
    for label, path, df in [("Parquet", LARGE_PARQUET, pqt),
                            ("CSV gzip", LARGE_CSV_GZ, gz),
                            ("CSV bzip2", LARGE_CSV_BZ2, bz)]:
        seconds, same = "not run (opt-in)", "not checked"
        if label == "Parquet" or RUN_LARGE_CSV:
            t0 = time.perf_counter()
            answer = large_query(source=df).collect()
            seconds = round(time.perf_counter() - t0, 2)
            if label == "Parquet":
                reference = answer
                same = "reference"
            else:
                assert_same_results(answer, reference)
                same = True
        format_rows.append({"format": label, "stored MB": mb(path),
                            "scan MB (estimate)": round(parquet_scan_mb, 1) if label == "Parquet" else mb(path),
                            "raw text MB decoded": "-" if label == "Parquet" else round(CSV_MB, 1),
                            "read partitions": df.rdd.getNumPartitions(),
                            "seconds (one run)": seconds, "same answer": same})
    print("\nSame no-filter aggregation, compressed formats:")
    print(pd.DataFrame(format_rows).to_string(index=False))

                       query seconds  payload MB row groups  vs raw CSV                          mechanism
CSV gzip/bzip2, decoded text not run     25609.2          -    baseline both decode all text; not disk I/O
          Parquet, no filter    1.02       257.0    125/125   100x less            column pruning: 5 of 61
Parquet, filter on DATEADDED    0.14         3.9      1/125 6,491x less      pushdown on the SORTED column
  Parquet, filter on SQLDATE    0.18       182.3     89/125   140x less     pushdown on an UNSORTED column



Same no-filter aggregation, compressed formats:
   format  stored MB  scan MB (estimate) raw text MB decoded  read partitions seconds (one run) same answer
  Parquet     4831.6               202.5                   -               38              0.52   reference
 CSV gzip     4014.2              4014.2             25609.2                1  not run (opt-in) not checked
CSV bzip2     2473.1              2473.1             25609.2               20  not run (opt-in) not checked


### Reading the table

**The ratios do not collapse at scale; they widen.** Column pruning alone takes 25 GiB down to
a few hundred megabytes, and the sorted-column filter takes it to single-digit megabytes — four
orders of magnitude, answered in a fraction of a second on a laptop. Ninety times the data, and
the query got *cheaper in absolute terms* than the CSV baseline of the small extract.

**The two filter rows are the point of the section.** Same file. Same footer. Same query shape.
Same number of rows in the answer.

* Filtering **`DATEADDED`**, the column the file is sorted by, skips 124 of 125 row groups. Each
  row group covers a few days and its recorded minimum and maximum exclude the target
  immediately.
* Filtering **`SQLDATE`**, the event date, skips far fewer — because GDELT collects events
  retrospectively, so nearly every row group contains a scattering of dates spanning about a
  decade. Row group 0 records a `SQLDATE` range of 2014-01-03 to 2024-01-09. A minimum and a
  maximum ten years apart exclude almost nothing, and the reader must open the group and check.

Both are Parquet. Both have complete footer statistics. Both plans show `PushedFilters`. One
saves ninety-eight percent of the read and the other saves thirty percent, and the whole of the
difference is *which column the rows happen to be sorted by*.

That is the practical form of the lesson: **when you choose the sort order of a table you are
choosing which future queries will be fast.** Statistics can only exclude a row group when the
values in it are clustered, and clustering is decided at write time, once, for every query
thereafter.

In [19]:
# The scratch copies are outside the repository.  Uncomment to reclaim the disk.
# shutil.rmtree(WORK, ignore_errors=True)
print("scratch left in place:", WORK, f"({mb(WORK)} MB)")

scratch left in place: /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777/ch03-parquet (1296.6 MB)


## Conclusion

The chapter says the gap between the first measurement and the last "is precisely the argument
of this section, made quantitative". The results table is that gap, and every step of it has a
name:

| Step | Mechanism | What it exploits |
|------|-----------|------------------|
| gzip CSV → bzip2 CSV | **splittable compression** | multiple tasks can decode one file, but every record must still be examined; codec speed also changes |
| CSV → Parquet, no filter | **column pruning**, plus encoding and compression | one column's values are stored contiguously, so five of sixty-one can be read alone |
| Parquet → Parquet sorted, with filter | **predicate pushdown** | the footer records min/max per column chunk, so row groups that cannot match are never read |
| sorted → partitioned, with filter | **partition pruning** | the value is in the directory name, so files are excluded before any footer is read |

Four things worth carrying out of this notebook:

1. **Instrument before you optimize.** `number of files read` is on every scan node in the Spark
   UI, and the footer arithmetic here is a dozen lines of `pyarrow` over metadata that ships
   inside the file.
2. **Know what your instrument measures.** `size of files read` sounds like the answer and is
   not: it reports split size and is blind to column pruning. A number you cannot interpret will
   mislead you more reliably than no number at all.
3. **Pushdown depends on layout, not merely on format.** Sorted Parquet skipped most of its row
   groups; the same rows shuffled skipped none, with an identical `PushedFilters` entry in the
   plan. `PushedFilters` proves the filter arrived, not that it helped. At 65.8 million rows the
   same file gave both answers at once, depending only on whether the filtered column was the
   one the rows were sorted by.
4. **Write-time choices are paid for on every read.** Row-group size sets the granularity of
   skipping, the codec is decoded by every query thereafter, and the file count can make a
   dataset simultaneously bigger and slower.

For Exercise 11 the same procedure transfers directly to the taxi data in
[3.8](03.08%20Taxi%20Trips%20with%20the%20DataFrame%20API.ipynb): declare the schema, write
Parquet partitioned by a low-cardinality date column, and point `footer_scan` at both.